In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.decomposition import PCA

from pandas import read_csv, DataFrame, concat

from numpy import arange, full, nan, size, shape, pi

from seaborn import heatmap

from os import path

from qiskit.circuit.library import z_feature_map, unitary_overlap, zz_feature_map

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

from qiskit.circuit.library import ZZFeatureMap
from qiskit import QuantumCircuit

import qnexus as qnx

from pytket.extensions.qiskit import qiskit_to_tk


DATA_PATH = path.join('..', 'data', "water_potability.csv")
RANDOM_SEED = 42
NUMBER_MUESTRAS_ENTRENAMIENTO = 12
USAR_MATRICES_PRECOMPUTADAS = True

# Get data

In [ ]:
data = read_csv(DATA_PATH)

data = data.dropna() # hay que limpiar los NaN, si no el normalizar no tiene sentido

data = data.sample(frac=1,\
                   random_state=RANDOM_SEED)

potable_data = data.loc[data['Potability']==1]

unpotable_data = data.loc[data['Potability']==0]

muestras = int((NUMBER_MUESTRAS_ENTRENAMIENTO/.8)/2)

data = concat([potable_data.iloc[0:muestras], \
               unpotable_data.iloc[0:muestras]])


data = data.sample(frac = 1, \
                   random_state=RANDOM_SEED)

'''

Hay que ver si se ponen nuevas features

'''

'''
El documento dice

Imputar valores faltantes (pH: 491 NaN; Sulfato: 781 NaN; Trihalometanos: 162 NaN) por mediana por clase.

Pero no sé si hay que hacer caso a eso xd

'''
potability = data["Potability"]

data = data.drop("Potability", axis = 1) # quitar potabilidad

# Pre-proccess data

In [ ]:
data_train, data_test, potability_train, potability_test = \
train_test_split(data, potability, test_size=.2, random_state=RANDOM_SEED)



#scaler = MinMaxScaler(feature_range=(0, 2 * pi))

scaler = StandardScaler()

data_train_scaled = scaler.fit_transform(data_train)
data_test_scaled = scaler.transform(data_test)

pca = PCA(n_components = 2, random_state=RANDOM_SEED)

data_train_scaled = pca.fit_transform(data_train_scaled)

data_test_scaled = pca.transform(data_test_scaled)

# SVM

In [ ]:
scores = ["precision", "recall"]

'''
Por si quiero quemar la compu

gamma = ["scale", "auto"]

gamma.extend(list(arange(start = .01, stop = 1, step = .005)))

C = arange(start=1, stop= 1000, step=100)

'''

C = [1, 10]

gamma = ["scale", "auto", .1]


tuned_parameters = [
    {"kernel": ["rbf"], "gamma": gamma, "C":C}
]

def refit_strategy(cv_results):
	
    cv_results_ = DataFrame(cv_results)

    best_index = cv_results_["mean_test_recall"].idxmax()

    return best_index

grid_search = GridSearchCV(
    SVC(), tuned_parameters, scoring=scores, refit=refit_strategy, cv=5, n_jobs=-1
)

grid_search.fit(data_train_scaled, potability_train)

# Test

In [ ]:
potability_prediction = grid_search.predict(data_test_scaled)

print(f'Confusion matrix: \n {confusion_matrix(potability_test, potability_prediction)}\n')
print(f'Accuracy = {accuracy_score(potability_test, potability_prediction):.2f}')
print(f'Clasification report: \n {classification_report(potability_test, \
potability_prediction, target_names=['Non-Potable' , 'Potable'])}')

In [ ]:
results = DataFrame(grid_search.cv_results_)


results.to_csv("laMambaNegra.csv")

print(f'Best params: {grid_search.best_params_}')

results[
    [
        "param_C",
        "param_gamma",
        "mean_test_precision",
        "mean_test_recall",
    ]
]

In [ ]:
heatmap(data.corr(), \
			 annot=True, \
			 robust=True, \
			 vmin= -1, \
			 vmax= 1)

# QSVM

# Nexus Login

In [ ]:
qnx.login()

# Create project

In [ ]:
project = qnx.projects.get_or_create(
    name="Water Potability QSVM"
)

qnx.context.set_active_project(project)

# Define Emulator

In [ ]:
backend = qnx.QuantinuumConfig(
    device_name="H2-Emulator"
)

# Quantum Kernel

In [ ]:
fm = z_feature_map(feature_dimension=data_train_scaled.shape[1])

train_size = data_train_scaled.shape[0]

kernel_matrix = np.eye(train_size)

num_shots = 100

programs = []
pairs = []

# 1) CREATE ALL CIRCUITS
for x1 in range(train_size):
    for x2 in range(x1 + 1, train_size):

        unitary1 = fm.assign_parameters(data_train_scaled[x1])
        unitary2 = fm.assign_parameters(data_train_scaled[x2])

        overlap_circ = unitary_overlap(unitary1, unitary2)
        overlap_circ.measure_all()

        tk_overlap = qiskit_to_tk(overlap_circ)

        program = qnx.circuits.upload(
            circuit=tk_overlap,
            name=f"kernel_{x1}_{x2}",
        )

        programs.append(program)
        pairs.append((x1, x2))


BATCH_SIZE = 100

for batch_start in range(0, len(programs), BATCH_SIZE):

    batch_programs = programs[batch_start:batch_start + BATCH_SIZE]
    batch_pairs = pairs[batch_start:batch_start + BATCH_SIZE]

    print(f"Processing batch {batch_start // BATCH_SIZE + 1}")
    print(f"Circuits: {len(batch_programs)}")


    # 2) COMPILE BATCH
    compile_job = qnx.start_compile_job(
        programs=batch_programs,
        backend_config=backend,
        optimisation_level=2,
        name=f"compile_kernel_batch_{batch_start}",
    )

    qnx.jobs.wait_for(compile_job)


    compiled = [
        result.get_output()
        for result in qnx.jobs.results(compile_job)
    ]


    # 3) EXECUTE BATCH
    execute_job = qnx.start_execute_job(
        programs=compiled,
        n_shots=[num_shots] * len(compiled),
        backend_config=backend,
        name=f"execute_kernel_batch_{batch_start}",
    )

    qnx.jobs.wait_for(execute_job)


    results = qnx.jobs.results(execute_job)


    # 4) FILL KERNEL MATRIX
    for pair, result in zip(batch_pairs, results):

        distribution = result.download_result().get_distribution()

        n_bits = len(next(iter(distribution)))
        zero_state = (0,) * n_bits

        prob_zero = distribution.get(zero_state, 0)

        x1, x2 = pair

        kernel_matrix[x1,x2] = prob_zero
        kernel_matrix[x2,x1] = prob_zero


    print("Batch completed")


print("Quantum Nexus Kernel Matrix:")
print(kernel_matrix)


# Quantum data Matrix 

In [ ]:
test_size = shape(data_test_scaled)[0]

'''
La matriz de prueba tiene forma test x train,
porque contiene la relación entre datos nuevos y datos de entrenamiento
'''

test_matrix = np.full((test_size, train_size), np.nan)

programs = []
pairs = []


# 1) CREATE ALL TEST CIRCUITS
for x1 in range(test_size):
    for x2 in range(train_size):

        unitary1 = fm.assign_parameters(data_test_scaled[x1])
        unitary2 = fm.assign_parameters(data_train_scaled[x2])

        overlap_circ = unitary_overlap(unitary1, unitary2)
        overlap_circ.measure_all()

        tk_overlap = qiskit_to_tk(overlap_circ)

        program = qnx.circuits.upload(
            circuit=tk_overlap,
            name=f"test_{x1}_{x2}",
        )

        programs.append(program)
        pairs.append((x1, x2))


BATCH_SIZE = 100

for batch_start in range(0, len(programs), BATCH_SIZE):

    batch_programs = programs[batch_start:batch_start + BATCH_SIZE]
    batch_pairs = pairs[batch_start:batch_start + BATCH_SIZE]

    print(f"Processing batch {batch_start // BATCH_SIZE + 1}")
    print(f"Circuits: {len(batch_programs)}")


    # Compile batch
    compile_job = qnx.start_compile_job(
        programs=batch_programs,
        backend_config=backend,
        optimisation_level=2,
        name=f"compile_test_batch_{batch_start}",
    )

    qnx.jobs.wait_for(compile_job)


    compiled = [
        result.get_output()
        for result in qnx.jobs.results(compile_job)
    ]


    # Execute batch
    execute_job = qnx.start_execute_job(
        programs=compiled,
        n_shots=[num_shots] * len(compiled),
        backend_config=backend,
        name=f"execute_test_batch_{batch_start}",
    )

    qnx.jobs.wait_for(execute_job)


    results = qnx.jobs.results(execute_job)


    # Fill test matrix
    for pair, result in zip(batch_pairs, results):

        distribution = result.download_result().get_distribution()

        prob_zero = distribution.get((0,0,0,0), 0)

        x1, x2 = pair

        test_matrix[x1, x2] = prob_zero


    print("Batch completed")


print("Pairs:", len(pairs))
print("Expected:", test_size * train_size)
print(test_matrix.shape)

# Test QSVM

In [ ]:
qsvm = SVC(kernel='precomputed')

qsvm.fit(kernel_matrix, potability_train)

potability_prediction = qsvm.predict(test_matrix)


print(f'Confusion matrix: \n {confusion_matrix(potability_test, potability_prediction)}\n')
print(f'Accuracy = {accuracy_score(potability_test, potability_prediction):.2f}')
print(f'Clasification report: \n {classification_report(potability_test, \
potability_prediction, target_names=['Non-Potable' , 'Potable'])}')


In [ ]:
import numpy as np
off_diag = kernel_matrix[~np.eye(kernel_matrix.shape[0], dtype=bool)]
print(f"Media: {off_diag.mean():.4f}, Std: {off_diag.std():.4f}")
print(f"Min: {off_diag.min():.4f}, Max: {off_diag.max():.4f}")
print(potability_prediction)
kernel_matrix

In [ ]:
print(potability_prediction)
print(potability_test)

In [ ]:
print(kernel_matrix)
print('Separando')
print(test_matrix)

In [ ]:
print(distribution)